In [2]:
from pathlib import Path
import sys
import math
import numpy as np
import jax
import jax.numpy as jnp

repo_root = Path('/home/atsuy/local_learning_coefficient')
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from sgld_utils import SGLDConfig, run_sgld
from utils import extrapolated_multiitemp_lambdahat

def quartic_loss(param, x, y):
    w = param["w"][0]
    return w**4

def exact_learning_coefficient_at_origin():
    # For K(w)=w^4, the RLCT / learning coefficient at the origin is 1/4.
    return 1 / 4

lambda_exact = exact_learning_coefficient_at_origin()
print(f"Python executable: {sys.executable}")
print(f"Theoretical learning coefficient for K(w)=w^4 at w=0: {lambda_exact}")

Python executable: /home/atsuy/Local_RLCT_calculater/.venv/bin/python
Theoretical learning coefficient for K(w)=w^4 at w=0: 0.25


## Numerical estimation with repository code

このリポジトリでは、SGLD で局所事後分布からサンプルした損失列 `loss_trace` を使って

$$\hat{\lambda} = (\mathrm{mean}(\mathrm{loss\_trace}) - L(w^*)) \, n \, \beta$$

を計算している。ここでは `w^*=0` なので `L(w^*) = 0` で、実装は `run_sgld` と `extrapolated_multiitemp_lambdahat` をそのまま使う。

In [3]:
def estimate_llc_with_repo(n, seed=0, epsilon=1e-4, gamma=1.0, num_steps=4000, batch_size=250):
    itemp = 1 / np.log(n)
    sgld_config = SGLDConfig(
        epsilon=epsilon,
        gamma=gamma,
        num_steps=num_steps,
        num_chains=1,
        batch_size=batch_size,
    )
    param_init = {"w": jnp.array([0.0])}
    x_train = jnp.zeros((n, 1))
    y_train = jnp.zeros((n, 1))
    rngkey = jax.random.PRNGKey(seed)

    loss_trace, _, _ = run_sgld(
        rngkey,
        quartic_loss,
        sgld_config,
        param_init,
        x_train,
        y_train,
        itemp=itemp,
        trace_batch_loss=True,
        compute_distance=False,
        compute_mala_acceptance=False,
        verbose=False,
    )

    loss_trace = np.asarray(loss_trace, dtype=float)
    init_loss = float(quartic_loss(param_init, x_train, y_train))
    lambdahat = (loss_trace.mean() - init_loss) * n * itemp
    lambdahat_extrapolated = extrapolated_multiitemp_lambdahat(loss_trace, n, itemp)
    return {
        "n": n,
        "itemp": itemp,
        "num_steps": num_steps,
        "mean_loss": loss_trace.mean(),
        "lambdahat": lambdahat,
        "lambdahat_extrapolated": float(lambdahat_extrapolated),
    }

results = [estimate_llc_with_repo(n) for n in [1000, 5000, 10000]]
for row in results:
    print(row)
print(f"\nTheoretical value: lambda = {lambda_exact}")

{'n': 1000, 'itemp': np.float64(0.14476482730108395), 'num_steps': 4000, 'mean_loss': np.float64(0.0015140769734382307), 'lambdahat': np.float64(0.21918509158033334), 'lambdahat_extrapolated': 0.19140514876715906}
{'n': 5000, 'itemp': np.float64(0.11740957114930958), 'num_steps': 4000, 'mean_loss': np.float64(0.00038221452173225334), 'lambdahat': np.float64(0.22437821541811168), 'lambdahat_extrapolated': 0.2350813959263081}
{'n': 10000, 'itemp': np.float64(0.10857362047581294), 'num_steps': 4000, 'mean_loss': np.float64(0.00020598426405240834), 'lambdahat': np.float64(0.2236445730921582), 'lambdahat_extrapolated': 0.22142797784321078}

Theoretical value: lambda = 0.25


推定値は有限サンプル・有限ステップなのでぴったり `0.25` にはならないが、`K(w)=w^4` の理論値 `1/4` の近くに来ることを確認できる。

In [4]:
# Optional: a somewhat longer run for a single n.
# If this feels slow in the notebook UI, skip this cell.
reference = estimate_llc_with_repo(n=5000, num_steps=8000)
reference

{'n': 5000,
 'itemp': np.float64(0.11740957114930958),
 'num_steps': 8000,
 'mean_loss': np.float64(0.00044443422470240746),
 'lambdahat': np.float64(0.2609041586319278),
 'lambdahat_extrapolated': 0.2245022764578193}

参考: 理論的には `K(w)=w^4` の原点での学習係数は `lambda = 1/4`。